# Проверка интернета на GPU-сервере

Запустите все ячейки в JupyterLab на **Student GPU Container**.

Проверяет доступ к сервисам DreamNarrative:
- **OpenRouter** — NSM через DeepSeek
- **HuggingFace** — скачивание Qwen
- **DNS / TCP** — базовая сеть

In [ ]:
import os
import socket
import platform
import subprocess
from datetime import datetime

print("Время:", datetime.now().isoformat())
print("Хост:", platform.node())
print("Python:", platform.python_version())
print("HTTP_PROXY:", os.environ.get("HTTP_PROXY") or os.environ.get("http_proxy") or "(нет)")
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy") or "(нет)")

In [ ]:
def check_tcp(host: str, port: int = 443, timeout: float = 5.0) -> tuple[bool, str]:
    try:
        socket.create_connection((host, port), timeout=timeout)
        return True, f"TCP {host}:{port} OK"
    except OSError as e:
        return False, f"TCP {host}:{port} FAIL: {e}"


def check_dns(host: str) -> tuple[bool, str]:
    try:
        ip = socket.gethostbyname(host)
        return True, f"DNS {host} -> {ip}"
    except OSError as e:
        return False, f"DNS {host} FAIL: {e}"


HOSTS = [
    ("8.8.8.8", "Google DNS (IP)"),
    ("google.com", "Google DNS"),
    ("openrouter.ai", "OpenRouter"),
    ("huggingface.co", "HuggingFace"),
]

rows = []
for host, label in HOSTS:
    if host.replace(".", "").isdigit():
        ok, msg = check_tcp(host, 53 if host == "8.8.8.8" else 443)
        rows.append((label, ok, msg))
    else:
        ok_dns, msg_dns = check_dns(host)
        ok_tcp, msg_tcp = check_tcp(host, 443)
        rows.append((label, ok_dns and ok_tcp, f"{msg_dns}; {msg_tcp}"))

print(f"{'Сервис':<16} {'Статус':<8} Детали")
print("-" * 70)
for label, ok, msg in rows:
    print(f"{label:<16} {'OK' if ok else 'FAIL':<8} {msg}")

In [ ]:
try:
    import httpx
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "httpx"])
    import httpx

URLS = [
    ("https://openrouter.ai/api/v1/models", "OpenRouter API"),
    ("https://huggingface.co", "HuggingFace"),
    ("https://pypi.org", "PyPI"),
]

print(f"{'Сервис':<18} {'HTTP':<6} {'Время':<8} Примечание")
print("-" * 75)

for url, label in URLS:
    try:
        t0 = __import__("time").time()
        r = httpx.get(url, timeout=15.0, follow_redirects=True)
        dt = __import__("time").time() - t0
        note = "OK" if r.status_code < 500 else f"server error {r.status_code}"
        ok = r.status_code < 500
        print(f"{label:<18} {r.status_code:<6} {dt:.2f}s    {note}")
    except Exception as e:
        print(f"{label:<18} {'—':<6} {'—':<8} FAIL: {e}")

In [ ]:
# OpenRouter с ключом из .env (если файл рядом с проектом)
from pathlib import Path

env_path = Path("../.env")
if not env_path.exists():
    env_path = Path("/home/student/work/dreamnarrative-backend/.env")

api_key = ""
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line.startswith("OPENROUTER_API_KEY="):
            api_key = line.split("=", 1)[1].strip()
        elif line.startswith("LLM_API_KEY=") and not api_key:
            api_key = line.split("=", 1)[1].strip()

if not api_key or "YOUR" in api_key:
    print("OPENROUTER_API_KEY не найден в .env — пропуск авторизованного запроса")
else:
    import httpx
    try:
        r = httpx.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={
                "model": "deepseek/deepseek-v4-flash",
                "max_tokens": 16,
                "messages": [{"role": "user", "content": "Say OK"}],
            },
            timeout=60.0,
        )
        print("OpenRouter chat:", r.status_code)
        if r.status_code == 200:
            print("Ответ:", r.json()["choices"][0]["message"]["content"][:200])
        else:
            print(r.text[:300])
    except Exception as e:
        print("OpenRouter chat FAIL:", e)

In [ ]:
# GPU + PyTorch (для локальной Qwen / SDXL)
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CC:", torch.cuda.get_device_capability(0))
        if hasattr(torch.cuda, "get_arch_list"):
            print("torch arch:", torch.cuda.get_arch_list())
        try:
            x = torch.ones(2, 2, device="cuda")
            x.mul_(2)
            torch.cuda.synchronize()
            print("CUDA kernel test: OK")
        except RuntimeError as e:
            print("CUDA kernel test: FAIL —", e)
            print("→ переустановите torch cu124 для Tesla V100")
except ImportError:
    print("torch не установлен — pip install -r requirements-gpu.txt")

In [ ]:
print("\n=== Рекомендация для .env ===\n")
try:
    import httpx
    r = httpx.get("https://openrouter.ai/api/v1/models", timeout=10)
    https_ok = r.status_code < 500
except Exception as e:
    https_ok = False
    print(f"OpenRouter недоступен: {e}")

if https_ok:
    print("Интернет есть → можно LLM_PROVIDER=openrouter")
else:
    print("Интернета нет (или только внутренняя сеть) → используйте:")
    print("  LLM_PROVIDER=local")
    print("  LLM_MODEL_ID=Qwen/Qwen3.5-4B")
    print("  LLM_FALLBACK_PROVIDER=local  # если оставляете openrouter")